In [1]:
import pandas as pd
import torch
print("GPU:",torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 4060 Ti


In [2]:
dataset = pd.read_csv("Social_Network_Ads.csv")
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [3]:
dataset = pd.get_dummies(dataset, dtype=int, drop_first=True)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [4]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [9]:
indep = dataset[['Age', 'EstimatedSalary',  'Gender_Male']]
depend = dataset[['Purchased']]

In [10]:
depend.value_counts()

Purchased
0            257
1            143
Name: count, dtype: int64

In [11]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(indep,depend,test_size=0.30,random_state=0)

In [32]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid = [
    {
        "solver": ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag'],
        "penalty": ['l2', None]
    },
    {
        "solver": ['liblinear'],
        "penalty": ['l1', 'l2']
    },
    {
        "solver": ['saga'],
        "penalty": ['l1', 'l2', 'elasticnet']
    }
]


grid = GridSearchCV(LogisticRegression(),parms_grid,refit=True,n_jobs=3,scoring='f1_weighted')
grid.fit(X_train,y_train)


C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\scipy\optimize\_linesearch.py:312: LineSearchWarning: The line search algorithm did not converge
  alpha_star, phi_star, old_fval, derphi_star = scalar_search_wolfe2(
C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\optimize.py:100: LineSearchWarning: The line search algorithm did not converge
  ret = line_search_wolfe2(


,estimator,LogisticRegression()
,param_grid,"{'penalty': ['l2'], 'solver': ['lbfgs', 'liblinear', ...]}"
,scoring,'f1_weighted'
,n_jobs,3
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [37]:
re = grid.cv_results_
print("The classification value for best parameter {}:".format(grid.best_params_))
table = pd.DataFrame.from_dict(re)
table

The classification value for best parameter {'penalty': 'l2', 'solver': 'newton-cg'}:


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.020810,0.000748,0.005401,0.000582,l2,lbfgs,"{'penalty': 'l2', 'solver': 'lbfgs'}",0.816207,0.775048,0.644599,0.927778,0.874356,0.807598,0.096542,3
1,0.002901,0.001114,0.006303,0.002250,l2,liblinear,"{'penalty': 'l2', 'solver': 'liblinear'}",0.503106,0.503106,0.591398,0.480769,0.520202,0.519716,0.037966,4
2,0.044434,0.002845,0.004401,0.000374,l2,newton-cg,"{'penalty': 'l2', 'solver': 'newton-cg'}",0.835985,0.802399,0.644599,0.927778,0.909115,0.823975,0.100806,1
3,0.003409,0.000570,0.005002,0.000707,l2,newton-cholesky,"{'penalty': 'l2', 'solver': 'newton-cholesky'}",0.835985,0.802399,0.644599,0.927778,0.909115,0.823975,0.100806,1
4,0.003502,0.000707,0.005102,0.000582,l2,sag,"{'penalty': 'l2', 'solver': 'sag'}",0.503106,0.503106,0.503106,0.480769,0.480769,0.494171,0.010943,5
5,0.003502,0.000316,0.004509,0.000457,l2,saga,"{'penalty': 'l2', 'solver': 'saga'}",0.503106,0.503106,0.503106,0.480769,0.480769,0.494171,0.010943,5


In [33]:
y_pred = grid.predict(X_test)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 0, 1, 1], dtype=int64)

In [38]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_pred,y_test)
print(cm)

[[74  8]
 [ 5 33]]


In [39]:
from sklearn.metrics import classification_report
report= classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.90      0.94      0.92        79
           1       0.87      0.80      0.84        41

    accuracy                           0.89       120
   macro avg       0.89      0.87      0.88       120
weighted avg       0.89      0.89      0.89       120

